In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
%%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
# from tpch import load_lineitem, load_orders, q12
DATA_ROOT = "/home/colinc/code/tpch/data/tpch_15"#"/home/jupyter/tpch-dbgen/data"#"/home/dias-benchmarks/tpch/data"
STORAGE_OPTS = {}



In [4]:

### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [5]:

### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [6]:
# %%time
# ### cell 2 ###
# # Filter, join, flag and aggregate in one pipeline
# total = (
#     lineitem[
#         (lineitem.L_RECEIPTDATE >= "1994-01-01")
#         & (lineitem.L_RECEIPTDATE <  "1995-01-01")
#         & (lineitem.L_COMMITDATE  <  "1995-01-01")
#         & (lineitem.L_SHIPDATE   <  "1995-01-01")
#         & (lineitem.L_SHIPDATE   <  lineitem.L_COMMITDATE)
#         & (lineitem.L_COMMITDATE <  lineitem.L_RECEIPTDATE)
#         &  lineitem.L_SHIPMODE.isin(["MAIL", "SHIP"])
#     ][["L_ORDERKEY", "L_SHIPMODE"]]
#     .merge(
#         orders[["O_ORDERKEY", "O_ORDERPRIORITY"]],
#         left_on="L_ORDERKEY", right_on="O_ORDERKEY"
#     )
#     .assign(
#         is_urgent     = lambda df: df.O_ORDERPRIORITY.isin(["1-URGENT", "2-HIGH"]),
#         is_not_urgent = lambda df: ~df.is_urgent
#     )
#     .groupby("L_SHIPMODE", as_index=False)
#     .agg(
#         g1=("is_urgent",     "sum"),
#         g2=("is_not_urgent", "sum")
#     )
# )


In [7]:
# total

In [8]:
%%time
##original
date1 = pd.Timestamp("1994-01-01")
date2 = pd.Timestamp("1995-01-01")
sel = (
    (lineitem.L_RECEIPTDATE < date2)
    & (lineitem.L_COMMITDATE < date2)
    & (lineitem.L_SHIPDATE < date2)
    & (lineitem.L_SHIPDATE < lineitem.L_COMMITDATE)
    & (lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE)
    & (lineitem.L_RECEIPTDATE >= date1)
    & ((lineitem.L_SHIPMODE == "MAIL") | (lineitem.L_SHIPMODE == "SHIP"))
)
flineitem = lineitem[sel]
jn = flineitem.merge(orders, left_on="L_ORDERKEY", right_on="O_ORDERKEY")

def g1(x):
    return ((x == "1-URGENT") | (x == "2-HIGH")).sum()

def g2(x):
    return ((x != "1-URGENT") & (x != "2-HIGH")).sum()

total = jn.groupby("L_SHIPMODE", as_index=False)["O_ORDERPRIORITY"].agg((g1, g2))
total = total.reset_index()  # reset index to keep consistency with pandas
# skip sort when groupby does sort already
# total = total.sort_values("L_SHIPMODE")
print(total)



   index L_SHIPMODE     g1      g2
0      0       MAIL  92997  139799
1      1       SHIP  93530  139811
CPU times: user 9.02 s, sys: 12.7 s, total: 21.7 s
Wall time: 18 s


In [ ]:
14.7, 24.5

In [ ]:
### cell 3 ###

total.info()